In [ ]:
%pip install SimpleITK==2.4.0 -q
%pip install numpy==1.26.4 -q
%pip install pyradiomics==3.1.0 -q
%pip install pandas==1.2.3 -q

In [1]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def featureExtractor(fileId):
  imagePath = './dataset/BraTS2021_Training_Data/%s/%s_flair.nii.gz' % (fileId, fileId)
  image = sitk.ReadImage(imagePath)
  maskPath ='./dataset/BraTS2021_Training_Data/%s/%s_kernel5_tumor.nii.gz' % (fileId, fileId)
  mask = sitk.ReadImage(maskPath)

  kernel = 5
  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 40000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName('ngtdm')

  featureMap = extractor.execute(image, mask, voxelBased=True)

  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      fileFolder = './dataset/ngtdm/kernel5-radius5/tumor/%s' % (fileId)
      if path.exists(fileFolder) == False:
        os.mkdir(fileFolder)
      sitk.WriteImage(featureValue, '%s/%s.nrrd' % (fileFolder, featureName))
      print('Computed %s, stored as "%s/%s.nrrd"' % (featureName, fileFolder, featureName))
    # else:
    #   print('%s: %s' % (featureName, featureValue))

monitorFilePath = './dataset/ngtdm/kernel5-radius5/tumor/ngtdm.monitor.csv'
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  fileId = row['file']
  print('Starting %s' % (fileId))
  featureExtractor(fileId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00002
Computed original_ngtdm_Busyness, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00002/original_ngtdm_Busyness.nrrd"
Computed original_ngtdm_Coarseness, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00002/original_ngtdm_Coarseness.nrrd"
Computed original_ngtdm_Complexity, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00002/original_ngtdm_Complexity.nrrd"
Computed original_ngtdm_Contrast, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00002/original_ngtdm_Contrast.nrrd"
Computed original_ngtdm_Strength, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00002/original_ngtdm_Strength.nrrd"
Starting BraTS2021_00003
Computed original_ngtdm_Busyness, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00003/original_ngtdm_Busyness.nrrd"
Computed original_ngtdm_Coarseness, stored as "./dataset/ngtdm/kernel5-radius5/tumor/BraTS2021_00003/original_ngtdm_Coarseness.nrrd"
Computed original_ngtdm_Complexity,

<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>